In [ ]:
import json, os, gc, time
from pathlib import Path
import numpy as np
import torch
from scipy.special import logsumexp, softmax
from safetensors import safe_open
from safetensors.torch import load_file, save_file
from transformers import AutoModelForCausalLM

In [ ]:
def carry(input_dir, output_dir):
    for source in input_dir.iterdir():
        target = output_dir / source.name
        target.unlink(missing_ok=True)
        target.symlink_to(os.path.relpath(source, output_dir), target_is_directory=source.is_dir())

def device_for(requested='auto'):
    if requested != 'auto':
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def load_model(path, device):
    model = AutoModelForCausalLM.from_pretrained(str(path), local_files_only=True,
        torch_dtype=torch.float32, attn_implementation='eager')
    model.config.use_cache = False
    for name, parameter in model.named_parameters():
        parameter.requires_grad_(name.startswith('model.layers.'))
    return model.to(device)

def reset_blocks(model, state):
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name.startswith('model.layers.'):
                parameter.copy_(state[name].float())

def block_state(model):
    return {name: parameter.detach().cpu().clone() for name, parameter in model.named_parameters()
            if name.startswith('model.layers.')}

In [ ]:
def train_steps(model, tokens, steps, rng, cfg, optimizer=None):
    if optimizer is None:
        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
            lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'], foreach=False)
    device = next(model.parameters()).device
    model.train()
    for _ in range(steps):
        starts = rng.integers(0, len(tokens) - cfg['sequence_length'] + 1, size=cfg['batch_size'])
        ids = np.stack([tokens[s:s + cfg['sequence_length']] for s in starts])
        batch = torch.tensor(ids, dtype=torch.long, device=device)
        optimizer.zero_grad(set_to_none=True)
        loss = model(input_ids=batch, labels=batch, use_cache=False).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    return optimizer

def language_metrics(model, evaluation_ids):
    model.eval()
    device = next(model.parameters()).device
    losses = []
    with torch.inference_mode():
        for row in evaluation_ids:
            batch = torch.tensor(row[None], dtype=torch.long, device=device)
            losses.append(float(model(input_ids=batch, labels=batch, use_cache=False).loss.cpu()))
    nll = float(np.mean(losses))
    return {'heldout_nll': nll, 'perplexity': float(np.exp(nll))}

In [ ]:
def marker_layout(model, maximum, rng):
    parameters = list(model.model.layers[0].named_parameters())
    sizes = np.array([p.numel() for _, p in parameters])
    offsets = np.r_[0, sizes.cumsum()]
    chosen = rng.choice(int(offsets[-1]), size=maximum, replace=False)
    groups = []
    for (name, parameter), left, right in zip(parameters, offsets[:-1], offsets[1:]):
        positions = np.flatnonzero((chosen >= left) & (chosen < right))
        if len(positions):
            groups.append({'name': name, 'positions': positions.tolist(), 'indices': (chosen[positions] - left).tolist()})
    return {'groups': groups, 'maximum': maximum, 'parameters_per_block': int(offsets[-1]),
            'blocks': len(model.model.layers)}

def fingerprint(model, layout):
    values = np.empty((len(model.model.layers), layout['maximum']), dtype=np.float32)
    with torch.no_grad():
        for j, layer in enumerate(model.model.layers):
            parameters = dict(layer.named_parameters())
            for group in layout['groups']:
                p = parameters[group['name']].detach().reshape(-1)
                ix = torch.tensor(group['indices'], dtype=torch.long, device=p.device)
                values[j, group['positions']] = p.index_select(0, ix).float().cpu().numpy()
    return values

def save_delta(model, base, path):
    # store updates, not rounded full weights; reconstruct before exact fingerprinting
    delta = {name: (parameter.detach().cpu() - base[name].float()).half().contiguous()
             for name, parameter in model.named_parameters() if name.startswith('model.layers.')}
    save_file(delta, str(path))

def apply_delta(model, base, path):
    with safe_open(str(path), framework='pt', device='cpu') as data, torch.no_grad():
        for name, parameter in model.named_parameters():
            if name.startswith('model.layers.'):
                parameter.copy_(base[name].float())
                parameter.add_(data.get_tensor(name).to(parameter.device, dtype=parameter.dtype))

def splice_blocks(model, base, donor_directory, split, ancestry):
    reset_blocks(model, base)
    for source in np.unique(ancestry):
        path = donor_directory / f'{split}_{source}.safetensors'
        with safe_open(str(path), framework='pt', device='cpu') as data, torch.no_grad():
            for layer in np.flatnonzero(ancestry == source):
                for suffix, parameter in model.model.layers[layer].named_parameters():
                    name = f'model.layers.{layer}.{suffix}'
                    parameter.add_(data.get_tensor(name).to(parameter.device, dtype=parameter.dtype))

def ancestry_path(rng, length, mean_tract):
    path = np.empty(length, dtype=int)
    path[0] = rng.integers(2)
    for j in range(1, length):
        path[j] = 1 - path[j - 1] if rng.random() < 1 / mean_tract else path[j - 1]
    return path

In [ ]:
def copying_cost(query, references, count, scale):
    states = references.reshape(-1, references.shape[-2], references.shape[-1])
    return (np.mean((states[:, :, :count] - query[None, :, :count]) ** 2, axis=-1) / scale[None]).T.astype(np.float64)

def viterbi(cost, lam):
    length, states = cost.shape
    penalty = lam * (1 - np.eye(states))
    score = cost[0].copy()
    back = np.zeros((length, states), dtype=int)
    for j in range(1, length):
        choices = score[:, None] + penalty
        back[j] = choices.argmin(axis=0)
        score = cost[j] + choices.min(axis=0)
    path = np.empty(length, dtype=int)
    path[-1] = score.argmin()
    for j in range(length - 1, 0, -1):
        path[j - 1] = back[j, path[j]]
    return path

def posterior(cost, lam, temperature):
    length, states = cost.shape
    emission = -cost / temperature
    transition = -lam * (1 - np.eye(states)) / temperature
    transition -= logsumexp(transition, axis=1, keepdims=True)
    alpha, beta = np.empty_like(cost), np.zeros_like(cost)
    alpha[0] = emission[0] - np.log(states)
    for j in range(1, length):
        alpha[j] = emission[j] + logsumexp(alpha[j - 1, :, None] + transition, axis=0)
    for j in range(length - 2, -1, -1):
        beta[j] = logsumexp(transition + emission[j + 1, None] + beta[j + 1, None], axis=1)
    return softmax(alpha + beta, axis=1)

def source_probabilities(probabilities, sources=2):
    return probabilities.reshape(*probabilities.shape[:-1], sources, -1).sum(axis=-1)

def boundary_f1(truth, prediction):
    true = set(np.flatnonzero(truth[1:] != truth[:-1]))
    called = set(np.flatnonzero(prediction[1:] != prediction[:-1]))
    return 2 * len(true & called) / (len(true) + len(called)) if true or called else 1.0